In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':', 1)
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'

# Remove rows where sample_title ends with '_POS'
metadata_df = metadata_df[~metadata_df['sample_title'].str.endswith('_POST')]

print(metadata_df.shape)
metadata_df.head()


(15, 16)


,sample_title,tissue,treatment,overall response,os,pfs,os event,pfs event,age,biomaterial provider,collection date,geo loc_name,Sex,disease,disease stage,race
samples,,,,,,,,,,,,,,,,
GSM7744827,R002,lympn node,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Progressive Disease,5.84804928131417,2.06981519507187,1,1,59,"Alberto J. Montero,alberto.montero@uhhospitals...",2017-08-25,"Cleveland, OH, USA",female,TNBC,stage IV,Black or African American
GSM7744829,R008_PRE,breast,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Partial Response,16.3285420944559,6.07802874743327,1,1,50,"Alberto J. Montero,alberto.montero@uhhospitals...",2017-11-08,"Cleveland, OH, USA",female,TNBC,stage IV,Not specified
GSM7744830,R009,breast,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Progressive Disease,6.50513347022587,2.0041067761807,1,1,49,"Alberto J. Montero,alberto.montero@uhhospitals...",2017-11-24,"Cleveland, OH, USA",female,TNBC,stage IV,White
GSM7744832,R010_PRE,unspecified metastatic site,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Partial Response,41.3305954825462,8.34496919917864,1,1,62,"Alberto J. Montero,alberto.montero@uhhospitals...",missing,"Cleveland, OH, USA",female,TNBC,stage IV,Not specified
GSM7744833,R012,lymph node,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Partial Response,11.0390143737166,11.0390143737166,1,0,70,"Alberto J. Montero,alberto.montero@uhhospitals...",2018-02-21,"Cleveland, OH, USA",female,TNBC,stage IV,White


In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"


df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)


df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_9012/3050319327.py:11: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,tissue,treatment,overall response,os,pfs,os event,pfs event,age,biomaterial provider,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM7744830,R009,breast,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Progressive Disease,6.50513347022587,2.0041067761807,1,1,49,"Alberto J. Montero,alberto.montero@uhhospitals...",...,122386,109,86,11,65983,27479,2790,148018,915,3592
GSM7744832,R010_PRE,unspecified metastatic site,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Partial Response,41.3305954825462,8.34496919917864,1,1,62,"Alberto J. Montero,alberto.montero@uhhospitals...",...,26509,21,21,10,25217,7024,545,44408,91,563
GSM7744833,R012,lymph node,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Partial Response,11.0390143737166,11.0390143737166,1,0,70,"Alberto J. Montero,alberto.montero@uhhospitals...",...,150255,47,32,29,72829,23246,2062,188954,152,827
GSM7744834,R013,lung,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Partial Response,4.92813141683778,4.92813141683778,1,0,45,"Alberto J. Montero,alberto.montero@uhhospitals...",...,90993,34,17,11,37498,22407,2816,52546,81,643
GSM7744836,R014_PRE,sternum,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Progressive Disease,18.7597535934292,1.90554414784394,1,1,36,"Alberto J. Montero,alberto.montero@uhhospitals...",...,120449,52,34,8,79308,27471,2619,125667,927,1909
GSM7744837,R015,lung,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Stable Disease,17.3141683778234,17.3141683778234,1,0,76,"Alberto J. Montero,alberto.montero@uhhospitals...",...,87919,38,24,20,61066,22862,2444,53643,170,1745
GSM7744838,R016,scapula,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Stable Disease,5.51950718685832,5.51950718685832,1,0,56,"Alberto J. Montero,alberto.montero@uhhospitals...",...,79495,18,12,4,51730,16296,1302,86908,372,2208
GSM7744839,R017,liver,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Partial Response,41.7577002053388,41.7577002053388,0,0,70,"Alberto J. Montero,alberto.montero@uhhospitals...",...,70902,46,38,9,56891,18190,1652,87786,158,1340
GSM7744841,R018_PRE,lymph node,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Partial Response,14.1601642710472,14.1601642710472,1,0,71,"Alberto J. Montero,alberto.montero@uhhospitals...",...,298202,57,10,9,115728,63829,6724,137814,251,3489


In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)

desc_data['data_summary'] = {
    'samples': df_combined.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}
print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated
